In [42]:
# =============================================================================
# Zelle 01 – Setup & Daten laden
# =============================================================================
# Notebook 05: Modelltraining Modell A (AP 3.4). Ablationsstudie ueber
# Feature-Sets x Modelltypen x Zielgroessen-Varianten.
# =============================================================================

import sys
sys.path.append('../src')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from viz_config_v2 import apply_store44_style, save_figure, COLOR_GOLD, COLOR_BLUE, COLOR_GREEN, COLOR_TEXT_MUTED, BOXPLOT_STYLE
from preprocessing import load_dataset, select_dataset, get_target, get_feature_set, FEATURE_SETS, TARGET_VARIANTS, DNResidualizer, NUMERISCHE_BASIS_SPALTEN, KATEGORIALE_SPALTEN

SEED = 42
apply_store44_style()

df = load_dataset("../data/processed/model_a_preprocessed.csv")

print(f"Datensatz geladen: {df.shape}")
print(f"Verfuegbare Feature-Sets: {list(FEATURE_SETS.keys())}")
print(f"Verfuegbare Zielgroessen-Varianten: {list(TARGET_VARIANTS.keys())}")

Datensatz geladen: (700, 44)
Verfuegbare Feature-Sets: ['original', 'original_no_kategorial', 'residual', 'combined']
Verfuegbare Zielgroessen-Varianten: ['binary', 'continuous', 'multilabel']


In [43]:
# =============================================================================
# Zelle 02 – Train/Test-Split (einmalig fixiert, fuer alle Kombinationen)
# =============================================================================
# Ein einziger Split ueber alle 700 Indizes, konsistent fuer alle
# Feature-Sets und Zielgroessen-Varianten verwendet (Vergleichbarkeit).
# Stratifizierung auf io_nio (binaeres Label), da dies das gemeinsame
# Bezugslabel fuer alle drei Zielgroessen-Varianten ist.
# =============================================================================

from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df["io_nio"], random_state=SEED
)

print(f"Train: {len(train_idx)} Zeilen, Test: {len(test_idx)} Zeilen")
print(f"\nIO/NIO-Verteilung Train:\n{df.loc[train_idx, 'io_nio'].value_counts(normalize=True).round(3)}")
print(f"\nIO/NIO-Verteilung Test:\n{df.loc[test_idx, 'io_nio'].value_counts(normalize=True).round(3)}")

Train: 560 Zeilen, Test: 140 Zeilen

IO/NIO-Verteilung Train:
io_nio
IO     0.739
NIO    0.261
Name: proportion, dtype: float64

IO/NIO-Verteilung Test:
io_nio
IO     0.736
NIO    0.264
Name: proportion, dtype: float64


In [53]:
# =============================================================================
# Zelle 03 – Modell-Registry (konservative Standard-Hyperparameter)
# =============================================================================
# Kein Tuning in diesem AP - Standardwerte mit Begruendung je Modell,
# siehe Diskussion. random_state=SEED durchgaengig fuer Reproduzierbarkeit.
# =============================================================================

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor

MODELLE_BINAER = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=SEED),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "SVC": SVC(C=1.0, kernel="rbf", random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=3, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=50, max_depth=3, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0, random_state=SEED),
    "XGBoost": XGBClassifier(
        n_estimators=50, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=2.0, random_state=SEED, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(
        n_estimators=50, num_leaves=7, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=30, random_state=SEED, verbose=-1),
    "GaussianNB": GaussianNB(),
    "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, early_stopping=True, random_state=SEED),
}

MODELLE_KONTINUIERLICH = {
    "Ridge": Ridge(random_state=SEED),
    "kNN": KNeighborsRegressor(n_neighbors=5),
    "SVR": SVR(C=1.0, kernel="rbf"),
    "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=3, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=50, max_depth=3, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0, random_state=SEED),
    "XGBoost": XGBRegressor(
        n_estimators=50, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=2.0, random_state=SEED),
    "LightGBM": LGBMRegressor(
        n_estimators=50, num_leaves=7, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=30, random_state=SEED, verbose=-1),
    "MLP": MLPRegressor(hidden_layer_sizes=(50,), max_iter=1000, early_stopping=True, random_state=SEED),
}

MODELLE_MULTILABEL = {
    "LogisticRegression": MultiOutputClassifier(LogisticRegression(max_iter=1000, random_state=SEED)),
    "kNN": MultiOutputClassifier(KNeighborsClassifier(n_neighbors=5)),
    "SVC": MultiOutputClassifier(SVC(C=1.0, kernel="rbf", random_state=SEED)),
    "RandomForest": MultiOutputClassifier(RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=3, random_state=SEED)),
    "HistGradientBoosting": MultiOutputClassifier(HistGradientBoostingClassifier(
        max_iter=50, max_depth=3, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0, random_state=SEED)),
    "XGBoost": MultiOutputClassifier(XGBClassifier(
        n_estimators=50, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=2.0, random_state=SEED, eval_metric="logloss")),
    "LightGBM": MultiOutputClassifier(LGBMClassifier(
        n_estimators=50, num_leaves=7, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=30, random_state=SEED, verbose=-1)),
    "GaussianNB": MultiOutputClassifier(GaussianNB()),
    "MLP": MultiOutputClassifier(MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, early_stopping=True, random_state=SEED)),
}

print(f"Modelle binaer: {list(MODELLE_BINAER.keys())} ({len(MODELLE_BINAER)})")
print(f"Modelle kontinuierlich: {list(MODELLE_KONTINUIERLICH.keys())} ({len(MODELLE_KONTINUIERLICH)})")
print(f"Modelle multilabel: {list(MODELLE_MULTILABEL.keys())} ({len(MODELLE_MULTILABEL)})")

Modelle binaer: ['LogisticRegression', 'kNN', 'SVC', 'RandomForest', 'HistGradientBoosting', 'XGBoost', 'LightGBM', 'GaussianNB', 'MLP'] (9)
Modelle kontinuierlich: ['Ridge', 'kNN', 'SVR', 'RandomForest', 'HistGradientBoosting', 'XGBoost', 'LightGBM', 'MLP'] (8)
Modelle multilabel: ['LogisticRegression', 'kNN', 'SVC', 'RandomForest', 'HistGradientBoosting', 'XGBoost', 'LightGBM', 'GaussianNB', 'MLP'] (9)


In [58]:
# =============================================================================
# Zelle 04 – Pipeline-Builder: baut fuer ein Feature-Set die korrekte,
# leakage-sichere Preprocessing-Pipeline (vor dem eigentlichen Modell)
# =============================================================================

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer

def baue_preprocessing_pipeline(feature_set_name):
    """
    Gibt einen sklearn-Transformer zurueck, der aus den Basis-Spalten
    (NUMERISCHE_BASIS_SPALTEN + KATEGORIALE_SPALTEN) das gewuenschte
    Feature-Set erzeugt - inkl. Skalierung. Leakage-sicher: DNResidualizer
    wird bei jedem Pipeline-Fit neu (nur auf Trainingsdaten) gefittet.
    """
    numerisch_transform = Pipeline([
        ("select_numerisch", FunctionTransformer(lambda X: X[NUMERISCHE_BASIS_SPALTEN])),
        ("scale", StandardScaler()),
    ])
    kategorial_transform = Pipeline([
        ("select_kategorial", FunctionTransformer(lambda X: X[KATEGORIALE_SPALTEN].astype(float))),
        ("scale", StandardScaler()),
    ])
    residual_transform = Pipeline([
        ("residualize", DNResidualizer(numerische_spalten=NUMERISCHE_BASIS_SPALTEN)),
        ("scale", StandardScaler()),
    ])

    if feature_set_name == "original":
        return FeatureUnion([("numerisch", numerisch_transform), ("kategorial", kategorial_transform)])
    elif feature_set_name == "original_no_kategorial":
        return numerisch_transform
    elif feature_set_name == "residual":
        return FeatureUnion([("residual", residual_transform), ("kategorial", kategorial_transform)])
    elif feature_set_name == "combined":
        return FeatureUnion([
            ("numerisch", numerisch_transform),
            ("residual", residual_transform),
            ("kategorial", kategorial_transform),
        ])
    else:
        raise ValueError(f"Unbekanntes Feature-Set: {feature_set_name}")

# --- Kurzer Test aller 4 Varianten ---
for fs in FEATURE_SETS:
    prep = baue_preprocessing_pipeline(fs)
    X_transformed = prep.fit_transform(df.loc[train_idx])
    print(f"{fs:25s} -> Shape nach Transformation: {X_transformed.shape}")

original                  -> Shape nach Transformation: (560, 11)
original_no_kategorial    -> Shape nach Transformation: (560, 9)
residual                  -> Shape nach Transformation: (560, 11)
combined                  -> Shape nach Transformation: (560, 20)


In [59]:
# =============================================================================
# Zelle 05 – Metrik-Hilfsfunktionen je Zielgroessen-Typ
# =============================================================================
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score, matthews_corrcoef,
    mean_absolute_error, mean_squared_error, r2_score, accuracy_score
)

def metriken_binaer(y_true, y_pred, y_pred_proba=None):
    metriken = {
        "f1": f1_score(y_true, y_pred, pos_label="NIO", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label="NIO", zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label="NIO", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }
    # KORREKTUR: roc_auc-Schluessel IMMER anlegen (auch ohne Wahrscheinlichkeiten),
    # sonst haben Train- und Val-Metriken unterschiedliche Schluesselmengen,
    # was beim Aggregieren zu KeyError fuehrt.
    if y_pred_proba is not None:
        y_true_num = (y_true == "NIO").astype(int)
        try:
            metriken["roc_auc"] = roc_auc_score(y_true_num, y_pred_proba)
        except ValueError:
            metriken["roc_auc"] = np.nan
    else:
        metriken["roc_auc"] = np.nan
    return metriken

def metriken_kontinuierlich(y_true, y_pred, schwellenwert=0):
    metriken = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
    }
    # Rueckuebersetzung in binaer: Wert < Schwelle -> NIO (negativer Sicherheitsabstand)
    y_true_bin = np.where(y_true < schwellenwert, "NIO", "IO")
    y_pred_bin = np.where(y_pred < schwellenwert, "NIO", "IO")
    metriken["f1_rueckuebersetzt"] = f1_score(y_true_bin, y_pred_bin, pos_label="NIO", zero_division=0)
    return metriken

def metriken_multilabel(y_true, y_pred):
    metriken = {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "subset_accuracy": accuracy_score(y_true, y_pred),
    }
    # Rueckuebersetzung in binaer: irgendein Label=1 -> NIO
    y_true_bin = np.where(np.array(y_true).sum(axis=1) > 0, "NIO", "IO")
    y_pred_bin = np.where(np.array(y_pred).sum(axis=1) > 0, "NIO", "IO")
    metriken["f1_rueckuebersetzt"] = f1_score(y_true_bin, y_pred_bin, pos_label="NIO", zero_division=0)
    return metriken

print("Metrik-Funktionen definiert: metriken_binaer, metriken_kontinuierlich, metriken_multilabel")

Metrik-Funktionen definiert: metriken_binaer, metriken_kontinuierlich, metriken_multilabel


In [60]:
# =============================================================================
# Zelle 06 – Ablationsschleife MIT Train-CV-Gap-Messung (Overfitting-Diagnose)
# =============================================================================
# Erweiterung ggue. erstem Durchlauf: zusaetzlich zur Validierungs-Metrik
# wird je Fold auch die Trainings-Metrik gemessen (Vorhersage auf X_tr
# selbst). Grosse Differenz Train-Val = Hinweis auf Overfitting
# (siehe Methodik-Glossar: Overfitting/Curse of Dimensionality).
# =============================================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore")

OUTPUT_CSV = "../reports/tables/05_ablation_results.csv"

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(train_idx, df.loc[train_idx, "io_nio"]))
fold_splits_idx = [(train_idx[tr_pos], train_idx[val_pos]) for tr_pos, val_pos in fold_splits]

MODELL_REGISTRY = {"binary": MODELLE_BINAER, "continuous": MODELLE_KONTINUIERLICH, "multilabel": MODELLE_MULTILABEL}
METRIK_FUNKTIONEN = {"binary": metriken_binaer, "continuous": metriken_kontinuierlich, "multilabel": metriken_multilabel}

ergebnisse = []
kombi_counter = 0
gesamt_kombis = sum(len(FEATURE_SETS) * len(MODELL_REGISTRY[tv]) for tv in TARGET_VARIANTS)
start_gesamt = time.time()

for target_variant in TARGET_VARIANTS:
    y_full, valid_mask = get_target(df, target_variant, respect_mask=True)

    for feature_set_name in FEATURE_SETS:
        for modell_name, modell in MODELL_REGISTRY[target_variant].items():
            kombi_counter += 1
            eintrag = {"zielgroesse": target_variant, "feature_set": feature_set_name, "modell": modell_name, "status": "ok"}
            fold_metriken_val, fold_metriken_train, fold_zeiten = [], [], []

            try:
                for tr_idx, val_idx in fold_splits_idx:
                    if target_variant == "continuous":
                        tr_idx_eff = tr_idx[valid_mask.loc[tr_idx].values]
                        val_idx_eff = val_idx[valid_mask.loc[val_idx].values]
                    else:
                        tr_idx_eff, val_idx_eff = tr_idx, val_idx

                    prep = baue_preprocessing_pipeline(feature_set_name)
                    X_tr = prep.fit_transform(df.loc[tr_idx_eff])
                    X_val = prep.transform(df.loc[val_idx_eff])
                    y_tr = y_full.loc[tr_idx_eff]
                    y_val = y_full.loc[val_idx_eff]

                    modell_fold = clone(modell)

                    # XGBoost-Fix: benoetigt numerisch kodierte Labels bei binaerer Klassifikation
                    label_kodiert = (target_variant == "binary" and modell_name == "XGBoost")
                    y_tr_fit = (y_tr == "NIO").astype(int) if label_kodiert else y_tr

                    t0 = time.time()
                    modell_fold.fit(X_tr, y_tr_fit)
                    fold_zeiten.append(time.time() - t0)

                    def _predict_zurueckuebersetzt(X):
                        pred_roh = modell_fold.predict(X)
                        if label_kodiert:
                            return np.where(pred_roh == 1, "NIO", "IO")
                        return pred_roh

                    y_pred_val = _predict_zurueckuebersetzt(X_val)
                    y_pred_train = _predict_zurueckuebersetzt(X_tr)

                    if target_variant == "binary":
                        y_proba_val = None
                        y_proba_train = None
                        if hasattr(modell_fold, "predict_proba"):
                            klassen = list(modell_fold.classes_)
                            idx_nio = klassen.index(1) if label_kodiert else klassen.index("NIO")
                            y_proba_val = modell_fold.predict_proba(X_val)[:, idx_nio]
                            y_proba_train = modell_fold.predict_proba(X_tr)[:, idx_nio]
                        m_val = metriken_binaer(y_val, y_pred_val, y_proba_val)
                        m_train = metriken_binaer(y_tr, y_pred_train, y_proba_train)
                    elif target_variant == "continuous":
                        m_val = metriken_kontinuierlich(y_val, y_pred_val)
                        m_train = metriken_kontinuierlich(y_tr, y_pred_train)
                    else:
                        m_val = metriken_multilabel(y_val.values, y_pred_val)
                        m_train = metriken_multilabel(y_tr.values, y_pred_train)

                    fold_metriken_val.append(m_val)
                    fold_metriken_train.append(m_train)

                for k in fold_metriken_val[0].keys():
                    werte_val = [fm[k] for fm in fold_metriken_val]
                    werte_train = [fm[k] for fm in fold_metriken_train]
                    eintrag[f"{k}_mean"] = np.nanmean(werte_val)
                    eintrag[f"{k}_std"] = np.nanstd(werte_val)
                    eintrag[f"train_{k}_mean"] = np.nanmean(werte_train)
                    gaps = [t - v for t, v in zip(werte_train, werte_val)]
                    eintrag[f"{k}_gap_mean"] = np.nanmean(gaps)

                eintrag["fit_time_sekunden_mean"] = np.mean(fold_zeiten)
                eintrag["n_folds_erfolgreich"] = len(fold_metriken_val)

            except Exception as e:
                eintrag["status"] = "fehlgeschlagen"
                eintrag["fehler"] = f"{type(e).__name__}: {str(e)[:200]}"

            ergebnisse.append(eintrag)
            pd.DataFrame(ergebnisse).to_csv(OUTPUT_CSV, index=False)
            print(f"[{kombi_counter}/{gesamt_kombis}] {target_variant:12s} {feature_set_name:25s} {modell_name:20s} -> {eintrag['status']}")

print(f"\nGesamtzeit: {time.time()-start_gesamt:.1f}s")
print(f"Erfolgreich: {(pd.DataFrame(ergebnisse)['status']=='ok').sum()} / {len(ergebnisse)}")

[1/104] binary       original                  LogisticRegression   -> ok
[2/104] binary       original                  kNN                  -> ok
[3/104] binary       original                  SVC                  -> ok
[4/104] binary       original                  RandomForest         -> ok
[5/104] binary       original                  HistGradientBoosting -> ok
[6/104] binary       original                  XGBoost              -> ok
[7/104] binary       original                  LightGBM             -> ok
[8/104] binary       original                  GaussianNB           -> ok
[9/104] binary       original                  MLP                  -> ok
[10/104] binary       original_no_kategorial    LogisticRegression   -> ok
[11/104] binary       original_no_kategorial    kNN                  -> ok
[12/104] binary       original_no_kategorial    SVC                  -> ok
[13/104] binary       original_no_kategorial    RandomForest         -> ok
[14/104] binary       original_no_

In [56]:
# =============================================================================
# Zelle 08 – Gesamtueberblick: beste Kombination je Zielgroessen-Typ
# =============================================================================
ergebnisse_df = pd.read_csv("../reports/tables/05_ablation_results.csv")
ergebnisse_df = ergebnisse_df[ergebnisse_df["status"] == "ok"].copy()

primary_metric = {"binary": "f1_mean", "continuous": "f1_rueckuebersetzt_mean", "multilabel": "f1_rueckuebersetzt_mean"}

print("=== Top 5 je Zielgroessen-Typ (nach Rueckfall-F1 bzw. nativer F1) ===\n")
for tv, metric_col in primary_metric.items():
    sub = ergebnisse_df[ergebnisse_df["zielgroesse"] == tv].sort_values(metric_col, ascending=False)
    print(f"--- {tv} (sortiert nach {metric_col}) ---")
    print(sub[["feature_set", "modell", metric_col, "fit_time_sekunden_mean"]].head(5).to_string(index=False))
    print()

print(f"Referenz: Bayes-Noise-Floor F1 = 0.447 (Notebook 03, Zelle 14)")
print(f"Referenz: Baseline uniform F1 = 0.351 (Notebook 03, Zelle 12)")

=== Top 5 je Zielgroessen-Typ (nach Rueckfall-F1 bzw. nativer F1) ===

--- binary (sortiert nach f1_mean) ---
           feature_set     modell  f1_mean  fit_time_sekunden_mean
original_no_kategorial GaussianNB 0.422805                0.004341
              original GaussianNB 0.420235                0.000595
              combined GaussianNB 0.411099                0.000805
original_no_kategorial        MLP 0.294678                0.050702
              residual GaussianNB 0.291758                0.000882

--- continuous (sortiert nach f1_rueckuebersetzt_mean) ---
           feature_set   modell  f1_rueckuebersetzt_mean  fit_time_sekunden_mean
              residual      MLP                 0.222346                0.170934
              original      MLP                 0.164618                0.075024
              combined LightGBM                 0.164556                0.007811
original_no_kategorial      kNN                 0.164165                0.001895
original_no_kategorial 

In [57]:
# =============================================================================
# Zelle 09 – Skalierungs-Check: sind alle Feature-Spalten vergleichbar skaliert?
# =============================================================================
prep = baue_preprocessing_pipeline("original")
X_check = prep.fit_transform(df.loc[train_idx])

spaltennamen = NUMERISCHE_BASIS_SPALTEN + KATEGORIALE_SPALTEN
X_check_df = pd.DataFrame(X_check, columns=spaltennamen)

print("=== Std-Abweichung je Spalte nach Preprocessing (Feature-Set 'original') ===")
print(X_check_df.std().round(3))

=== Std-Abweichung je Spalte nach Preprocessing (Feature-Set 'original') ===
schneckendrehzahl        1.001
massedurchsatz           1.001
massetemperatur          1.001
massedruck               1.001
duesenspalt              1.001
abzugsgeschwindigkeit    1.001
kalibrierdruck_mbar      1.001
kuehlwassertemperatur    1.001
mfr_charge               1.001
wandtyp_einwandig        0.449
mechanismus_Vakuum       0.272
dtype: float64
